# 04 | Portfolio Risk Analytics and Contribution Decomposition

**Research question.** What did the diversification and risk diagnostics show for the **17 actually held equities** relative to the earlier 11-stock allocation, at the same equity exposure?

**Reference snapshot:** NextWise, 5 November 2026 at 12:29; NAV EUR 50,926.12, cash EUR 18,379.64, equities EUR 32,546.48. **Risk sample:** 252 adjusted daily returns ending 4 November 2026. Holdings and sector mapping are from the uploaded operating notebook and earlier NextWise confirmations. These data are historical and **not live**.

The weights use 4-Nov close prices (including EUR-converted Oslo equities) and are **rescaled** to the 5-Nov NextWise aggregate equity value. Accordingly, **individual weights are estimated**, not exact 5-Nov intraday broker weights. No trades are submitted.

This notebook is a direct scholarly refactor of the final STEP 44B, not a new proprietary alpha strategy.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Run from the repository root, or from within notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
OUTPUT = ROOT / "reports" / "research_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT.resolve())
print("Data directory:", DATA.resolve())

In [ ]:
PRICE_FILE = DATA / "prices_eur_2026-11-04.csv"
if not PRICE_FILE.is_file():
    raise FileNotFoundError(
        f"Missing {PRICE_FILE}. Copy your previously validated EUR-adjusted "
        "price panel into data/ (or update PRICE_FILE). Do not commit private "
        "or third-party price datasets without reviewing distribution rights."
    )
prices = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
prices.index = pd.to_datetime(prices.index).normalize()
prices = prices.apply(pd.to_numeric, errors="coerce").sort_index()
if prices.index.has_duplicates or prices.columns.has_duplicates:
    raise ValueError("Duplicate timestamps or ticker identifiers")
if prices.isna().any().any() or (prices <= 0).any().any():
    raise ValueError("Missing or nonpositive values in the EUR price panel")
if prices.shape[0] < 254:
    raise ValueError("At least 254 prices required for a 252-return risk window")
returns = prices.pct_change(fill_method=None).dropna()
print(f"{prices.shape[1]} securities, {prices.shape[0]} dates; last = {prices.index[-1].date()}")

## 1. Declare historical execution quantities and the broker NAV snapshot

In [ ]:
ASOF = pd.Timestamp("2026-11-04")
if prices.index[-1] != ASOF:
    raise ValueError("Expected clean 4-Nov-2026 historical cutoff; do not use later observations")
NAV, CASH, EQUITY = 50926.12, 18379.64, 32546.48
if abs(NAV-CASH-EQUITY)>0.01:
    raise AssertionError("NextWise NAV reconciliation failed")
EXPOSURE=EQUITY/NAV
quantities=pd.Series({'AI.PA': 8, 'ENX.PA': 1, 'HEIA.AS': 29, 'TTE.PA': 26, 'G.MI': 52, 'SRG.MI': 336, 'KPN.AS': 462, 'SAN.PA': 25, 'PRY.MI': 18, 'ML.PA': 62, 'CAP.PA': 15, 'OR.PA': 4, 'CS.PA': 47, 'EDP.LS': 420, 'RACE.MI': 5, 'EQNR.OL': 49, 'UCB.BR': 7},dtype=float)
sectors={'AI.PA': 'Materials', 'ENX.PA': 'Financials', 'HEIA.AS': 'Consumer Staples', 'TTE.PA': 'Energy', 'G.MI': 'Financials', 'SRG.MI': 'Utilities', 'KPN.AS': 'Telecommunications', 'SAN.PA': 'Healthcare', 'PRY.MI': 'Industrials', 'ML.PA': 'Consumer Discretionary', 'CAP.PA': 'Technology', 'OR.PA': 'Consumer Staples', 'CS.PA': 'Financials', 'EDP.LS': 'Utilities', 'RACE.MI': 'Consumer Discretionary', 'EQNR.OL': 'Energy', 'UCB.BR': 'Healthcare'}
assert len(quantities)==17
if not set(quantities.index).issubset(prices.columns):
    raise ValueError("Missing one or more held securities in EUR prices")
mark = quantities * prices.loc[ASOF,quantities.index]
marked_values = mark / mark.sum() * EQUITY
old_names=list({'AI.PA': 8, 'ENX.PA': 1, 'HEIA.AS': 29, 'TTE.PA': 26, 'G.MI': 52, 'SRG.MI': 336, 'KPN.AS': 462, 'SAN.PA': 25, 'PRY.MI': 18, 'ML.PA': 62, 'CAP.PA': 15})
old_mark = quantities.loc[old_names] * prices.loc[ASOF,old_names]
benchmark_values = old_mark / old_mark.sum() * EQUITY
r252 = returns.tail(252)
print(f"Positions: {len(quantities)} | equity exposure: {EXPOSURE:.4%} | max mark weight: {marked_values.max()/NAV:.3%}")

## 2. Risk definitions

For equity weights \(w_E\) summing to one and annualized return covariance \(\Sigma\):

- \(\sigma_E=\sqrt{w_E^	op\Sigma w_E}\), \(\sigma_{NAV}=e\sigma_E\), where \(e\) is equity exposure.
- \(DR=(\sum_i w_{E,i}\sigma_i)/\sigma_E\).
- \(HHI_E=\sum_i w_{E,i}^2\), and \(N_{eff}=1/HHI_E\).
- Historical daily CVaR (95%) is the mean loss among returns below the sample 5th percentile.
- Historical maximum drawdown uses the simulated constant-weight daily NAV return path.

**Risk units:** volatility is annualized; daily CVaR is *not* annualized. The cash return is assumed zero for the historical simulation.

In [ ]:
def risk_metrics(values, ret):
    tickers=list(values.index)
    w_eq=values/values.sum()
    w_nav=w_eq*EXPOSURE
    r=ret[tickers]
    covariance=r.cov()*252
    vol_eq=float(np.sqrt(w_eq.values@covariance.values@w_eq.values))
    vol_nav=EXPOSURE*vol_eq
    individual=r.std(ddof=1)*np.sqrt(252)
    dr=float((w_eq@individual)/vol_eq)
    daily=(r@w_nav).to_numpy()
    cutoff=np.quantile(daily,0.05)
    cvar=float(-np.mean(daily[daily<=cutoff]))
    path=np.r_[1.0,np.cumprod(1.0+daily)]
    mdd=float(np.min(path/np.maximum.accumulate(path)-1))
    hhi=float(np.sum(w_eq.values**2))
    return {"Exposure_pct":EXPOSURE*100,"Vol_Equity_pct":100*vol_eq,
            "Vol_NAV_pct":100*vol_nav,"DR":dr,"HHI_Equity":hhi,
            "Effective_N":1/hhi,"CVaR95_daily_pct":100*cvar,
            "MDD_pct":100*mdd,"Max_Stock_NAV_pct":float(w_nav.max())*100}
compare=pd.DataFrame({"Benchmark_11_same_exposure":risk_metrics(benchmark_values,r252),
                      "Portfolio_17":risk_metrics(marked_values,r252)})
compare["Delta_17_minus_11"]=compare["Portfolio_17"]-compare["Benchmark_11_same_exposure"]
print(compare.round(4).to_string())

## 3. Horizon sensitivity: 63, 126 and 252 observations

These windows overlap and therefore **do not constitute out-of-sample validation**.

In [ ]:
stability=pd.DataFrame([{"Trading_Days":n,**risk_metrics(marked_values,returns.tail(n))} for n in (63,126,252)])
print(stability[["Trading_Days","DR","Vol_Equity_pct","Vol_NAV_pct","CVaR95_daily_pct","MDD_pct"]].round(4).to_string(index=False))

## 4. Euler decomposition of portfolio volatility

For NAV weights \(w\): \(RC_i=w_i(\Sigma w)_i/\sigma_P\). Signed relative contributions sum to 100%. **Negative contributions are possible** when a position has negative marginal covariance with the portfolio, but they do not guarantee safe hedging in future regimes.

In [ ]:
w_nav=marked_values/NAV
cov=r252[quantities.index].cov()*252
v=float(np.sqrt(w_nav.values@cov.values@w_nav.values))
rc_component=w_nav.values*(cov.values@w_nav.values)/v
rc=pd.DataFrame({"Ticker":quantities.index,
                 "Sector":[sectors[t] for t in quantities.index],
                 "NAV_weight_pct":100*w_nav.values,
                 "Risk_contribution_pct":100*rc_component/v,
                 "Component_vol_pp":100*rc_component}).sort_values("Risk_contribution_pct",ascending=False)
assert np.isclose(rc["Risk_contribution_pct"].sum(),100,atol=1e-6)
print(rc.round(3).to_string(index=False))

## 5. Sector and concentration checks (internal rules only)

The code below uses *internal* 5% per-name and 15% sector thresholds. The simplified UCITS check **is not a substitute** for legal UCITS analysis or for the official competition's scoring method.

In [ ]:
sector_nav = (pd.DataFrame({"Sector":[sectors[t] for t in quantities.index],
                                "Value":marked_values.values})
              .groupby("Sector")["Value"].sum()/NAV).sort_values(ascending=False)
checks={"Name weight <= 5% NAV":bool((w_nav<=0.05+1e-12).all()),
        "Sector weight <= 15% NAV":bool((sector_nav<=0.15+1e-12).all()),
        "Simplified UCITS 10/40":bool(w_nav.max()<=0.10+1e-12 and w_nav[w_nav>0.05].sum()<=0.40+1e-12)}
print((100*sector_nav).round(3).to_string())
print("Checks:",checks)

## 6. Audit outputs and interpretation

Results can differ slightly from the original notebook if adjusted price histories, forex conventions, or exact sample periods are modified. Reconcile any discrepancy rather than replacing historical reported figures with new estimates.

In [ ]:
compare.to_csv(OUTPUT/"04_risk_comparison.csv")
stability.to_csv(OUTPUT/"04_risk_horizon_sensitivity.csv",index=False)
rc.to_csv(OUTPUT/"04_euler_risk_contributions.csv",index=False)
sector_nav.rename("NAV_sector_weight").to_csv(OUTPUT/"04_sector_exposure.csv")
print("Saved four risk outputs to",OUTPUT)

## Research limitations

- Covariance matrices and CVaR rely on recent historical observations; return distributions are nonstationary.
- Sector mappings are provisional and may differ from NextWise's official sector taxonomy.
- Neither the simulated DR nor this risk report directly reconstructs official Euronext points.
- No broker-execution ledger, complete FX settlement series or transaction costs are embedded in the notebooks.
- An authentic forward validation requires future observations that were not used for model selection.